# Reconnaissance du pays d'une pièce de 2 €

Classification de la face nationale en **23 pays** avec YOLO (Ultralytics).

| Section | Contenu |
|---|---|
| 0 | Configuration (`TRAIN = False` pour évaluer un modèle déjà entraîné sans réentraîner) |
| 1 | Exploration du dataset |
| 2 | Entraînement avec **rotation 0-360°** |
| 3 | Évaluation sur pièces **droites** et **tournées** (conditions webcam) |
| 4 | Baseline DINOv2 + régression logistique (optionnelle) |
| 5 | Comparaison |
| 6 | Test sur une image et export ONNX pour `camera.py` |

> Une pièce devant une webcam n'est presque jamais droite : le score qui compte est celui sur les **pièces tournées**.

## 0. Configuration

In [5]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

ROOT = Path.cwd()  # le notebook est lancé depuis dataset/
DATA = ROOT / "dataset_split"
TRAIN_DIR, VAL_DIR = DATA / "train", DATA / "val"
RUNS = ROOT / "runs"

TRAIN = True               # False : charge RUNS/RUN_NAME/weights/best.pt sans réentraîner
RUN_BASELINE = False       # True : compare avec DINOv2 + régression logistique (~10 min sur CPU)
RUN_NAME = "coin-cls-rot"  # dossier du run dans runs/ (l'ancien modèle sans rotation est dans runs/coin-cls)

# Réglages pour un entraînement sur CPU (Ryzen 5 PRO 5650U, 6 cœurs, 32 Go de RAM, pas de GPU NVIDIA)
MODEL = "yolo11n-cls.pt"   # alternatives : "yolo26n-cls.pt", "yolo11s-cls.pt" (~3x plus lent)
IMGSZ = 256
EPOCHS = 40                # la rotation rend la tâche plus difficile : un peu plus d'epochs
BATCH = 32
WORKERS = 4
SEED = 42

DEVICE = 0 if torch.cuda.is_available() else "cpu"
torch.set_num_threads(6)
print("torch", torch.__version__, "| device:", torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

IMG_EXT = {".jpg", ".jpeg", ".png"}
CLASSES = sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_dir())


def list_images(split_dir):
    """Retourne [(chemin, index_classe)] pour un dossier train/ ou val/."""
    return [(img, i) for i, c in enumerate(CLASSES)
            for img in sorted((split_dir / c).iterdir()) if img.suffix.lower() in IMG_EXT]


train_items, val_items = list_images(TRAIN_DIR), list_images(VAL_DIR)
y_true = np.array([y for _, y in val_items])
print(f"{len(CLASSES)} classes | train {len(train_items)} | val {len(val_items)}")

torch 2.14.1+cpu | device: CPU
23 classes | train 3473 | val 648


In [ ]:
# Deux versions de la validation, préparées une fois pour toutes :
#  - "droite"  : les images telles quelles
#  - "tournée" : chaque image tournée d'un angle aléatoire fixe (fond blanc), identique pour tous les modèles
def load_small(path, max_side=512):
    im = Image.open(path).convert("RGB")
    im.thumbnail((max_side, max_side))
    return im


rng = np.random.default_rng(SEED)
val_upright = [load_small(p) for p, _ in val_items]
val_angles = rng.uniform(0, 360, len(val_items))
val_rotated = [im.rotate(a, fillcolor=(255, 255, 255)) for im, a in zip(val_upright, val_angles)]
VAL_SETS = {"droite": val_upright, "tournée": val_rotated}

## 1. Exploration du dataset

In [ ]:
from collections import Counter

train_counts = Counter(CLASSES[y] for _, y in train_items)
val_counts = Counter(CLASSES[y] for _, y in val_items)

x = np.arange(len(CLASSES))
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(x, [train_counts[c] for c in CLASSES], label="train")
ax.bar(x, [val_counts[c] for c in CLASSES], bottom=[train_counts[c] for c in CLASSES], label="val")
ax.set_xticks(x, CLASSES, rotation=60, ha="right")
ax.set_ylabel("images"); ax.set_title("Images par pays"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Un exemple par pays (vérifier qu'il n'y a pas de face commune : carte de l'Europe)
fig, axes = plt.subplots(4, 6, figsize=(14, 10))
for ax, cls in zip(axes.flat, CLASSES):
    img = random.choice([p for p, y in train_items if CLASSES[y] == cls])
    ax.imshow(Image.open(img).convert("RGB")); ax.set_title(cls, fontsize=9); ax.axis("off")
for ax in axes.flat[len(CLASSES):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 2. Entraînement YOLO avec rotation

⚠️ En **classification**, Ultralytics **ignore** le paramètre `degrees` (il ne sert qu'à la détection).
`RotationTrainer` ajoute donc lui-même une `RandomRotation` 0-360° en tête des transformations d'entraînement.
La validation n'est pas tournée par Ultralytics : la section 3 mesure les deux cas.

Autres augmentations :
- `fliplr=0` : pas de miroir (une pièce inversée n'existe pas, le texte deviendrait faux) ;
- `hsv_v` / `hsv_s` : variations d'éclairage et de reflets ;
- `erasing` : masque une partie de l'image (reflet, doigt...).

`cache="ram"` décode les images une seule fois (les photos font jusqu'à 2300 px).

In [ ]:
from torchvision import transforms as T
from ultralytics import YOLO
from ultralytics.models.yolo.classify import ClassificationTrainer


class RotationTrainer(ClassificationTrainer):
    """Ajoute une rotation aléatoire 0-360° aux images d'entraînement (fond blanc dans les coins)."""

    def build_dataset(self, img_path, mode="train", batch=None):
        dataset = super().build_dataset(img_path, mode, batch)
        if mode == "train":
            dataset.torch_transforms.transforms.insert(0, T.RandomRotation(180, fill=(255, 255, 255)))
        return dataset


BEST = RUNS / RUN_NAME / "weights" / "best.pt"

if TRAIN:
    model = YOLO(MODEL)
    model.train(
        trainer=RotationTrainer,
        data=str(DATA),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        workers=WORKERS,
        cache="ram",
        patience=12,
        device=DEVICE,
        seed=SEED,
        project=str(RUNS),
        name=RUN_NAME,
        exist_ok=True,
        fliplr=0.0,
        hsv_h=0.01,
        hsv_s=0.5,
        hsv_v=0.4,
        erasing=0.3,
    )
    BEST = Path(model.trainer.best)

assert BEST.exists(), f"Poids introuvables : {BEST} (mettre TRAIN = True ou corriger RUN_NAME)"
print("Poids utilisés :", BEST)

In [ ]:
# Courbes d'entraînement
res = np.genfromtxt(BEST.parent.parent / "results.csv", delimiter=",", names=True, deletechars="")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
a1.plot(res["epoch"], res["train/loss"], label="train"); a1.plot(res["epoch"], res["val/loss"], label="val")
a1.set_title("loss"); a1.set_xlabel("epoch"); a1.legend()
a2.plot(res["epoch"], res["metrics/accuracy_top1"], label="top-1"); a2.plot(res["epoch"], res["metrics/accuracy_top5"], label="top-5")
a2.set_title("accuracy validation (pièces droites)"); a2.set_xlabel("epoch"); a2.legend()
plt.tight_layout(); plt.show()

## 3. Évaluation du modèle YOLO

In [6]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
BEST = Path("runs\coin-cls\weights\best.pt")
best = YOLO(BEST)
assert list(best.names.values()) == CLASSES, "ordre des classes différent"


def yolo_probs(images):
    return np.stack([r.probs.data.cpu().numpy()
                     for r in best.predict(images, imgsz=IMGSZ, device=DEVICE, stream=True, verbose=False)])


def scores(probs):
    top1 = (probs.argmax(1) == y_true).mean()
    top5 = np.mean([y in np.argsort(p)[-5:] for y, p in zip(y_true, probs)])
    return top1, top5


YOLO_PROBS = {name: yolo_probs(images) for name, images in VAL_SETS.items()}
for name, probs in YOLO_PROBS.items():
    print(f"YOLO pièces {name:8} top-1 = {scores(probs)[0]:.3f} | top-5 = {scores(probs)[1]:.3f}")

<>:2: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<>:2: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
C:\Users\tangu\AppData\Local\Temp\ipykernel_19792\2349303838.py:2: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
  BEST = Path("runs\coin-cls\weights\best.pt")


NameError: name 'YOLO' is not defined

In [ ]:
def plot_confusion(y_pred, title):
    fig, ax = plt.subplots(figsize=(11, 10))
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred, labels=range(len(CLASSES)), display_labels=CLASSES,
        normalize="true", values_format=".1f", cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(title)
    plt.xticks(rotation=70, ha="right"); plt.tight_layout(); plt.show()


yolo_pred = YOLO_PROBS["tournée"].argmax(1)
plot_confusion(yolo_pred, "YOLO, pièces tournées : matrice de confusion (normalisée par pays réel)")
print(classification_report(y_true, yolo_pred, labels=range(len(CLASSES)), target_names=CLASSES, digits=3))

In [ ]:
def show_errors(probs, images, n=12):
    pred = probs.argmax(1)
    errors = np.flatnonzero(pred != y_true)
    print(f"{len(errors)} erreurs sur {len(y_true)}")
    if not len(errors):
        return
    sel = np.random.default_rng(SEED).choice(errors, min(n, len(errors)), replace=False)
    rows = int(np.ceil(len(sel) / 6))
    fig, axes = plt.subplots(rows, 6, figsize=(15, 3 * rows), squeeze=False)
    for ax, i in zip(axes.flat, sel):
        ax.imshow(images[i]); ax.axis("off")
        ax.set_title(f"vrai : {CLASSES[y_true[i]]}\nprédit : {CLASSES[pred[i]]} ({probs[i].max():.0%})",
                     fontsize=8, color="crimson")
    for ax in axes.flat[len(sel):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()


show_errors(YOLO_PROBS["tournée"], val_rotated)

In [ ]:
# Test-time augmentation : moyenne des prédictions sur 4 orientations (option --tta de camera.py)
tta = np.mean([yolo_probs([im.rotate(a, fillcolor=(255, 255, 255)) for im in val_rotated])
               for a in (0, 90, 180, 270)], axis=0)
print(f"YOLO pièces tournées, sans TTA : {scores(YOLO_PROBS['tournée'])[0]:.3f}")
print(f"YOLO pièces tournées, avec TTA : {scores(tta)[0]:.3f}  -> utiliser --tta dans camera.py si c'est mieux")

## 6. Test sur une image et export ONNX

Remplacer `IMAGE` par une photo. L'export ONNX produit le fichier utilisé par `camera.py` :
`python camera.py --weights runs/<RUN_NAME>/weights/best.onnx` (ajouter `--tta` si la section 3 montre que ça aide).

In [ ]:
IMAGE = val_items[0][0]  # <- remplacer par le chemin d'une photo

r = best.predict(str(IMAGE), imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
plt.imshow(Image.open(IMAGE).convert("RGB")); plt.axis("off")
plt.title(f"{r.names[r.probs.top1]} ({r.probs.top1conf:.0%})"); plt.show()
for i, conf in zip(r.probs.top5, r.probs.top5conf):
    print(f"  {r.names[i]:15} {conf:.1%}")

In [1]:
onnx_path = best.export(format="onnx", imgsz=IMGSZ)
print("Exporté :", onnx_path)
print(f"Lancer : python camera.py --weights {Path(onnx_path).relative_to(ROOT)}")

NameError: name 'best' is not defined